# 23 — Paper 3: Table 4–6, Fig. 6–8 และ attributable visits (Bayesian NB / NB-BYM2, ปีงบประมาณ)

โน้ตบุ๊กนี้สร้างผลทั้งหมดของ manuscript Paper 3 ส่วน **2.7–2.10 และ 3.4–3.8** ในการรันครั้งเดียว

| ผลลัพธ์ | Manuscript | ไฟล์ใน `OUTDIR` |
|---|---|---|
| Rate ratio ตามเขตสัมผัส (PM2.5 Q4 vs Q1, ท้ายลม, ริมถนน) แบบ fiscal-year strata | Sect. 3.4 | `RR_exposure_zones.csv` |
| IRR ภายในตำบล (lag 0, lag 1, lag 0–1) และ IRR ระหว่างตำบล × 4 กลุ่มโรค | **Table 5**, Fig. 6 | `Table5_IRR.csv`, `Fig6_IRR.png` |
| เปรียบเทียบแบบจำลอง (PSIS-LOO) + การทำนายนอกตัวอย่าง (LODO, ช่วงสุดท้าย) | **Table 4** | `Table4_model_comparison.csv` |
| Effect modification ตามตำแหน่งท้ายลมและกลุ่มประชากรริมถนน | Sect. 3.6, Fig. 7 | `EM_effect_modification.csv` |
| Predicted SIR, exceedance probability, ตำบลเป้าหมาย | Sect. 3.8, **Table 6**, Fig. 8 | `Table6_priority_tambons.csv`, `Risk_all_tambons.csv`, `Fig8_predicted_risk.png` |
| จำนวนครั้งที่มารับบริการที่เกิดจาก PM2.5 เกิน 15 และ 5 µg/m³ (สมการ 3) | Sect. 3.8 | `Attributable_visits.csv` |
| การลู่เข้าของ MCMC ทุกแบบจำลอง | Sect. 3.5 | `convergence.csv` |
| ข้อความตัวเลขสำหรับวางใน manuscript | 3.5–3.7 | `manuscript_numbers.txt` |

**แบบจำลอง** (เหมือน Notebook 20 แต่เพิ่ม ERA5 และใช้ช่วงข้อมูลดาวเทียมเท่านั้น)

$$Y_{it}\sim\text{NegBin}(\mu_{it},\alpha),\qquad
\ln\mu_{it}=\ln E_{it}+\beta_0+\beta_W\,x^W_{it}+\beta_B\,x^B_i+\mathbf z_t^\top\boldsymbol\gamma+\delta_{m(t)}+\tau_{fy(t)}+\phi_i$$

- $E_{it}$ = ประชากร WorldPop 2020 × อัตรามารับบริการเฉลี่ยของจังหวัด (ต่อเดือน)
- $x^W_{it}=(x_{it}-\bar x_i)/10$ ส่วนภายในตำบล (ผลหลัก) · $x^B_i=(\bar x_i-\bar x)/10$ ส่วนระหว่างตำบล (Begg & Parides 2003)
- $\mathbf z_t$ = อุณหภูมิ ความชื้นสัมพัทธ์ ฝนรวม รายเดือนจาก ERA5 (จุด 14.50°N 101.00°E, standardised)
- $\delta_m$ = เดือนในรอบปี · **$\tau_{fy}$ = ปีงบประมาณ (ต.ค.–ก.ย.)** · ทั้งสองแบบ zero-sum
- $\phi_i$ = BYM2 (Riebler et al. 2016) บน Queen contiguity — เขียนแบบ hierarchical centring (Gelfand et al. 1995)
- ไม่มีตัวแปร fire-hotspot หรือ COVID-19 · ผลช่วงโควิดถูกดูดซับโดย $\tau_{fy}$

**ปีงบประมาณ:** ปีในข้อมูล HDC ดิบเป็นปีปฏิทิน (ตรวจแล้ว 26 ต.ค. 2569) โน้ตบุ๊กนี้แปลงเป็นปีงบ พ.ศ. = ปีปฏิทิน พ.ศ. + 1 สำหรับเดือน ต.ค.–ธ.ค.
ในช่วงหลัก (ม.ค. 2565 – พ.ย. 2567 ตัด ธ.ค.) ปีงบ 2565 มี 9 เดือน, 2566 และ 2567 มีปีละ 11 เดือน, 2568 มี 2 เดือน (ต.ค.–พ.ย. 2567)

**เวลารัน:** 15 แบบจำลอง MCMC + cross-validation ≈ 2–4 ชั่วโมงบน Colab CPU (ประมาณการ) · ส่วน rate ratio, risk map และ attributable visits ใช้เวลาไม่กี่นาที (ใช้ posterior ของแบบจำลองหลัก ไม่ต้องรันเพิ่ม)
ทุกแบบจำลองบันทึกเป็น `.nc` ใน `OUTDIR/fits/` ทันทีที่รันเสร็จ — ถ้า Colab หลุด ให้ `Run all` ใหม่ ระบบจะข้ามแบบจำลองที่รันแล้ว
ตั้ง `FAST = True` เพื่อทดสอบก่อน (ห้ามใช้ผลรายงาน)

## 0 · CONFIG

In [ ]:
# ===== Repository data (data/processed/) =====
# Opened from GitHub in Colab: the repository is cloned so that data/processed/ is available.
# Run locally: start Jupyter in the repository (or its notebooks/ folder).
# If data/processed/ is not found, the notebook falls back to the original Google Drive paths.
import os, subprocess
REPO_URL = "https://github.com/songwutsa/paper3-pm25-acute-illness-saraburi"
USE_REPO = True          # False = use the original Google Drive paths only
REPO_DIR = None
for _d in (os.getcwd(), os.path.dirname(os.getcwd()), "/content/paper3-pm25-acute-illness-saraburi"):
    if os.path.isdir(os.path.join(_d, "data", "processed")):
        REPO_DIR = _d; break
if USE_REPO and REPO_DIR is None and os.path.isdir("/content"):
    REPO_DIR = "/content/paper3-pm25-acute-illness-saraburi"
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=False)
REPO_DATA = os.path.join(REPO_DIR, "data", "processed") if REPO_DIR else ""
USE_REPO_DATA = bool(USE_REPO and REPO_DATA and os.path.isdir(REPO_DATA))

def repo_file(name):
    """Path of a file in data/processed/ (matched by file name), or None."""
    p = os.path.join(REPO_DATA, os.path.basename(str(name))) if USE_REPO_DATA else ""
    return p if p and os.path.exists(p) else None

print("Repository data:", REPO_DATA if USE_REPO_DATA else "not found - using Google Drive paths")

In [ ]:
# ---- ไฟล์ข้อมูล (ชื่อไฟล์จะถูกค้นหาอัตโนมัติใน DATA_DIR และ MyDrive ถ้าไม่พบใน path ที่ระบุ) ----
DATA_DIR        = "/content/drive/MyDrive/PM25_Saraburi_notebooks"
PM_PANEL_CSV    = "pm25_tambon_monthly_fy2565_2569.csv"      # Notebook 06: ตำบล × เดือน (ต.ค. 2564 – ก.ย. 2569)
BOUNDARY_GPKG   = "sat_pm25_tambon_summary.gpkg"             # Notebook 06: ขอบเขต 111 ตำบล + ประชากร WorldPop 2020
HDC_MONTHLY_CSV = "000_research/11HealthImpactAssessment_SW/0data/hdc_saraburi_pm25_disease_tambon_monthly_2565_2569.csv"
DIW_CSV         = "3-ColabProjects/Paper2/02_Data/processed/diw_facilities_geocoded.csv"   # 295 โรงงาน (lat, lon)
ROADSIDE_CSV    = "roadside_tambon.csv"                       # Notebook 08
ERA5_CACHE_DIR  = "ERA5_cache"                                # era5_<lat>_<lon>_<year>_<hash>.csv.gz (รายชั่วโมง)
ERA5_POINT      = (14.50, 101.00)

# ---- ช่วงวิเคราะห์หลัก (เฉพาะเดือนที่มี PM2.5 ดาวเทียม) ----
START, END      = "2022-01", "2024-11"
DROP_DECEMBER   = True                 # ธ.ค. บันทึกไม่ครบใน HDC (15–35% ของเดือนปกติ)
PM_COL          = "pm25_sat_pop"       # PM2.5 ดาวเทียมถ่วงประชากร
PM_UNIT         = 10                   # IRR ต่อ 10 µg/m³

# ---- กลุ่มโรค (ชื่อคอลัมน์ตามสคริปต์ดึง HDC รายเดือน) ----
GROUPS = {
    "Respiratory":    ["COPD", "Asthma", "Pneumonia", "Influenza", "Acute pharyngitis", "Chronic rhinitis", "Bronchitis"],
    "Cardiovascular": ["Ischemic heart diseases", "Cerebrovascular diseases (Stroke)"],
    "Eye":            ["กลุ่มโรคตาอักเสบ"],
    "Skin":           ["กลุ่มโรคผิวหนังอักเสบ"],
}
LAGS = {"Lag 0": [0], "Lag 1": [1], "Lag 0–1": [0, 1]}    # lag 0–1 = ค่าเฉลี่ยของเดือนนี้และเดือนก่อน

# ---- ตัวแปรตำแหน่งแหล่งกำเนิดและริมถนน (Sect. 2.4) ----
DOWNWIND_SECTOR = (270.0, 315.0)       # ทิศจากโรงงานไปตำบล (CPF 90–135° กลับทิศ) — manuscript 2.4
DOWNWIND_KM     = 5.0
ROADSIDE_COL    = "share_500m_MAIN"    # สัดส่วนประชากรภายใน 500 ม. จากทางหลวง 1 หรือ 2

# ---- การทำนายนอกตัวอย่าง (Sect. 2.9) ----
TRAIN_END       = "2023-11"            # leave-last-period-out: เรียน ม.ค. 2565 – พ.ย. 2566 → ทำนาย ม.ค.–พ.ย. 2567

# ---- MCMC (Sect. 2.8) ----
FAST = False                           # True = ทดสอบเร็ว (ห้ามรายงานผล)
DRAWS, TUNE, CHAINS = (100, 150, 2) if FAST else (300, 300, 4)
TARGET_ACCEPT, TARGET_ACCEPT_RETRY = 0.90, 0.97
SEED = 2026
RUN_TABLE5, RUN_TABLE4, RUN_CV = True, True, True
RUN_RR, RUN_RISK = True, True

# ---- Risk mapping และ attributable visits (Sect. 2.10) ----
EXCEED_HI, EXCEED_LO = 0.8, 0.2       # Richardson et al. (2004)
AV_THRESHOLDS = {"Thai standard": 15, "WHO guideline": 5}
DISTRICT_COL = None                    # ชื่อคอลัมน์ชื่ออำเภอใน BOUNDARY_GPKG (None = หาอัตโนมัติ; ไม่พบ → ใช้รหัสอำเภอ)
HOSPITAL_CODES = []                    # รหัส 6 หลักของ 12 ตำบลที่มีโรงพยาบาลรัฐ (ถ้าใส่ จะมีคอลัมน์ hospital ใน Table 6)
SAVE_TO_DRIVE = True

In [ ]:
# ===== Use data/processed/ from the repository when available (the Drive paths above are the fallback) =====
if USE_REPO_DATA:
    DATA_DIR = REPO_DATA
    for _v in ("PM_PANEL_CSV", "SAT_MONTHLY_CSV", "BOUNDARY_GPKG", "HDC_MONTHLY_CSV", "DIW_CSV", "ROADSIDE_CSV"):
        if globals().get(_v): globals()[_v] = os.path.basename(globals()[_v])
    if "ROADS_CANDIDATES" in globals():
        ROADS_GEOJSON = "osm_highway_1_2.geojson"; ROADS_CANDIDATES = [ROADS_GEOJSON]
    if "ERA5_CACHE_DIR" in globals():
        ERA5_CACHE_DIR = os.path.join(REPO_DATA, "ERA5_cache")   # hourly ERA5 2022–2024 (Open-Meteo archive); other years are downloaded
    SAVE_TO_DRIVE = False                                  # outputs go to the working folder, not data/processed/
    print("Using repository data in", DATA_DIR)

In [ ]:
import importlib.util, subprocess, sys
need = {"pymc": "pymc", "arviz": "arviz", "numpyro": "numpyro", "libpysal": "libpysal", "esda": "esda",
        "netCDF4": "netCDF4", "h5py": "h5py", "h5netcdf": "h5netcdf", "pyproj": "pyproj", "openpyxl": "openpyxl"}
missing = [p for m, p in need.items() if importlib.util.find_spec(m) is None]
if missing:
    print("ติดตั้ง:", missing); subprocess.run(f"{sys.executable} -m pip -q install " + " ".join(missing), shell=True)

In [ ]:
import os, re, glob, json, time, warnings, importlib.util
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt, requests
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr
from libpysal.weights import Queen
from pyproj import Geod
import pymc as pm, arviz as az, pytensor.tensor as pt
warnings.filterwarnings("ignore"); pd.set_option("display.width", 180)
UTM, GEOD = "EPSG:32647", Geod(ellps="WGS84")

IN_COLAB = os.path.isdir("/content")
if IN_COLAB and DATA_DIR.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
DATA_DIR = os.environ.get("PAPER3_DATA_DIR", DATA_DIR)
MYDRIVE = "/content/drive/MyDrive" if IN_COLAB else DATA_DIR
OUTDIR = os.path.join(DATA_DIR, "paper3", "table4_5") if SAVE_TO_DRIVE else "./table4_5"
FITDIR = os.path.join(OUTDIR, "fits" + ("_FAST" if FAST else ""))
os.makedirs(FITDIR, exist_ok=True)

def find(p, must=True):
    """path เต็ม → ใช้เลย · path สัมพัทธ์ → DATA_DIR, MyDrive · ไม่พบ → ค้นชื่อไฟล์ใน DATA_DIR (ลึก 4 ชั้น)"""
    if os.path.isabs(p) and os.path.exists(p): return p
    for b in (DATA_DIR, MYDRIVE, "."):
        if os.path.exists(os.path.join(b, p)): return os.path.join(b, p)
    name = os.path.basename(p)
    for root, dirs, files in os.walk(DATA_DIR):
        if root[len(DATA_DIR):].count(os.sep) >= 4: dirs[:] = []
        if name in files: return os.path.join(root, name)
    if must: raise FileNotFoundError(f"ไม่พบ {p} — แก้ path ใน CONFIG")
    return None

print("PyMC", pm.__version__, "| ArviZ", az.__version__, "| OUTDIR =", OUTDIR)

## 1 · ข้อมูล: ขอบเขตตำบล, PM2.5, ผู้ป่วย HDC รายเดือน
- ปีงบประมาณคำนวณจากเดือนจริง: `fy = ปีปฏิทิน พ.ศ. + (เดือน ≥ 10)` และตรวจเทียบกับคอลัมน์ `fy_be` ของสคริปต์ดึง HDC (ถ้ามี)
- ตำบลที่ใช้ = ตำบลที่มีทั้งในขอบเขตและใน HDC (109 จาก 111 ตำบล)

In [ ]:
code6 = lambda s: s.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True).str.zfill(6).str[:6]

BND = gpd.read_file(find(BOUNDARY_GPKG)); BND["a_code"] = code6(BND["a_code"])
_dc = DISTRICT_COL or next((c for c in BND.columns if c.lower() in
       ("district", "district_en", "district_name", "amphoe", "amphoe_en", "amp_name", "amp_name_e", "adm2_en", "name_2")), None)
BND["district_name"] = BND[_dc].astype(str) if _dc else BND.a_code.str[:4]
BND = BND[["a_code", "name", "district_name", "pop", "geometry"]].drop_duplicates("a_code").reset_index(drop=True)
print("คอลัมน์ชื่ออำเภอ:", _dc or "ไม่พบ → ใช้รหัสอำเภอ 4 หลัก")

PM = pd.read_csv(find(PM_PANEL_CSV), encoding="utf-8-sig", dtype={"a_code": str}); PM["a_code"] = code6(PM.a_code)
PM["ym"] = PM.ym.astype(str).str[:7]
if PM_COL not in PM: raise KeyError(f"ไม่มีคอลัมน์ {PM_COL} ใน {PM_PANEL_CSV}: {list(PM.columns)}")

ALL_DIS = [d for v in GROUPS.values() for d in v]
H = pd.read_csv(find(HDC_MONTHLY_CSV), encoding="utf-8-sig", dtype={"a_code": str})
H.columns = [c.strip() for c in H.columns]
ALIAS = {"Ischemic heart disease": "Ischemic heart diseases", "Stroke": "Cerebrovascular diseases (Stroke)",
         "Eye inflammation": "กลุ่มโรคตาอักเสบ", "Dermatitis": "กลุ่มโรคผิวหนังอักเสบ",
         "r1_1": "COPD", "r1_2": "Asthma", "r1_3": "Pneumonia", "r1_4": "Influenza", "r1_5": "Acute pharyngitis",
         "r1_6": "Chronic rhinitis", "r1_7": "Bronchitis", "r2_1": "Ischemic heart diseases",
         "r2_2": "Cerebrovascular diseases (Stroke)", "r3": "กลุ่มโรคตาอักเสบ", "r4": "กลุ่มโรคผิวหนังอักเสบ"}
H = H.rename(columns={k: v for k, v in ALIAS.items() if k in H.columns and v not in H.columns})
miss = [d for d in ALL_DIS if d not in H]
if miss: raise KeyError(f"ไม่พบคอลัมน์โรค {miss} ใน HDC — คอลัมน์ที่มี: {list(H.columns)}")
H["a_code"] = code6(H.a_code)
if "ym" not in H:                                   # ปีในไฟล์ดิบเป็นปีปฏิทิน
    y = pd.to_numeric(H.year_be); y = np.where(y > 2400, y - 543, y)
    H["ym"] = [f"{int(a):04d}-{int(b):02d}" for a, b in zip(y, H.month)]
H["ym"] = H.ym.astype(str).str[:7]
for d in ALL_DIS: H[d] = pd.to_numeric(H[d], errors="coerce").fillna(0)
t_ = pd.to_datetime(H.ym)
H["fy"] = t_.dt.year + 543 + (t_.dt.month >= 10).astype(int)
if "fy_be" in H:
    bad = (pd.to_numeric(H.fy_be) != H.fy).sum()
    print(f"ตรวจปีงบเทียบคอลัมน์ fy_be ของไฟล์ HDC: ไม่ตรง {bad} แถว" + (" ✓" if bad == 0 else " ⚠ ตรวจไฟล์ HDC"))
H = H.groupby(["a_code", "ym", "fy"], as_index=False)[ALL_DIS].sum()
H = H[H.a_code.str[:2] == "19"]
TAM_CODES = sorted(set(BND.a_code) & set(H.a_code))
print(f"ตำบลในขอบเขต {BND.a_code.nunique()} | ใน HDC {H.a_code.nunique()} | ใช้วิเคราะห์ {len(TAM_CODES)} "
      f"| ไม่มีใน HDC: {', '.join(BND.loc[~BND.a_code.isin(H.a_code), 'name'])}")

## 2 · ERA5 รายเดือน (อุณหภูมิ, ความชื้นสัมพัทธ์, ฝนรวม)
อ่านไฟล์รายชั่วโมงใน `ERA5_cache` (รูปแบบเดียวกับ Notebook 12–13) ถ้าไม่มีปีใดจะดึงจาก Open-Meteo ERA5 archive แล้วบันทึกลง cache

In [ ]:
def era5_hourly(lat, lon, year):
    cdir = find(ERA5_CACHE_DIR, must=False) or os.path.join(DATA_DIR, ERA5_CACHE_DIR)
    os.makedirs(cdir, exist_ok=True)
    hits = sorted(glob.glob(os.path.join(cdir, f"era5_{lat:.2f}_{lon:.2f}_{year}_*.csv.gz")), key=os.path.getmtime)
    if hits: return pd.read_csv(hits[-1], parse_dates=["time"])
    end = min(pd.Timestamp(f"{year}-12-31"), pd.Timestamp.today().normalize() - pd.Timedelta(days=6))
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=180, params=dict(
        latitude=lat, longitude=lon, start_date=f"{year}-01-01", end_date=f"{end:%Y-%m-%d}", models="era5",
        hourly="temperature_2m,relative_humidity_2m,precipitation", timezone="Asia/Bangkok"))
    r.raise_for_status(); h = r.json()["hourly"]
    df = pd.DataFrame(h).rename(columns={"time": "time"}); df["time"] = pd.to_datetime(df.time)
    df.to_csv(os.path.join(cdir, f"era5_{lat:.2f}_{lon:.2f}_{year}_openmeteo.csv.gz"), index=False)
    return df

yrs = range(int(START[:4]), int(END[:4]) + 1)
ERA = pd.concat([era5_hourly(*ERA5_POINT, y) for y in yrs], ignore_index=True)
ERA["ym"] = ERA.time.dt.strftime("%Y-%m")
MET = ERA.groupby("ym").agg(temp=("temperature_2m", "mean"), rh=("relative_humidity_2m", "mean"),
                            precip=("precipitation", "sum"), n_hours=("temperature_2m", "count")).reset_index()
MET = MET[(MET.ym >= START) & (MET.ym <= END)]
short = MET[MET.n_hours < 24 * 27]
if len(short): print("⚠ เดือนที่ ERA5 ไม่ครบ:", short.ym.tolist())
print(MET.describe().round(1).loc[["mean", "min", "max"]])

## 3 · ตัวแปรระดับตำบล: ท้ายลมโรงงาน (downwind) และกลุ่มประชากรริมถนน
- **Downwind:** จุดภายในตำบล (representative point) อยู่ห่างโรงงาน DIW อย่างน้อย 1 แห่ง ≤ `DOWNWIND_KM` และอยู่ในทิศ `DOWNWIND_SECTOR` จากโรงงานนั้น
- **Roadside:** none (สัดส่วน = 0) · low / high แบ่งที่มัธยฐานของสัดส่วนที่ > 0 ในตำบลที่วิเคราะห์

In [ ]:
TAM = BND[BND.a_code.isin(TAM_CODES)].sort_values("a_code").reset_index(drop=True)
N = len(TAM); TIDX = {c: i for i, c in enumerate(TAM.a_code)}
pt_ = TAM.to_crs(UTM).representative_point().to_crs(4326); TLAT, TLON = pt_.y.values, pt_.x.values

DIW = pd.read_csv(find(DIW_CSV)); DIW = DIW.dropna(subset=["lat", "lon"])
lo_, hi_ = DOWNWIND_SECTOR
dw = np.zeros(N, bool); dmin = np.full(N, np.inf)
for la, lo in zip(DIW.lat, DIW.lon):
    az_, _, d = GEOD.inv(np.full(N, lo), np.full(N, la), TLON, TLAT); az_ = az_ % 360; d = d / 1000
    dw |= (d <= DOWNWIND_KM) & (az_ >= lo_) & (az_ <= hi_); dmin = np.minimum(dmin, d)
TAM["downwind"] = dw.astype(int); TAM["dist_fac_km"] = dmin

RS = pd.read_csv(find(ROADSIDE_CSV), encoding="utf-8-sig", dtype={"a_code": str}); RS["a_code"] = code6(RS.a_code)
TAM = TAM.merge(RS[["a_code", ROADSIDE_COL]].rename(columns={ROADSIDE_COL: "road_share"}), on="a_code", how="left")
TAM["road_share"] = TAM.road_share.fillna(0)
med_ = TAM.loc[TAM.road_share > 0, "road_share"].median()
TAM["road_grp"] = np.where(TAM.road_share == 0, "none", np.where(TAM.road_share <= med_, "low", "high"))
print(f"DIW {len(DIW)} โรงงาน | downwind {TAM.downwind.sum()} จาก {N} ตำบล (sector {lo_:.1f}–{hi_:.1f}°, ≤ {DOWNWIND_KM} km)")
print(f"roadside ({ROADSIDE_COL}): {TAM.road_grp.value_counts().to_dict()} | มัธยฐานสัดส่วนที่ > 0 = {med_:.1%}")

## 4 · ตาราง ตำบล × เดือน, ค่า lag และโครงข่าย BYM2

In [ ]:
P = PM[PM.a_code.isin(TAM_CODES)][["a_code", "ym", PM_COL]].rename(columns={PM_COL: "pm"})
P["per"] = pd.PeriodIndex(P.ym, freq="M")
def pm_lag(k):
    s = P.assign(per=P.per + k)[["a_code", "per", "pm"]]; s["ym"] = s.per.astype(str); return s[["a_code", "ym", "pm"]]
PMX = None
for name_, ks in LAGS.items():
    parts = [pm_lag(k).rename(columns={"pm": f"_l{k}"}) for k in ks]
    m_ = parts[0]
    for p_ in parts[1:]: m_ = m_.merge(p_, on=["a_code", "ym"])
    m_[name_] = m_[[f"_l{k}" for k in ks]].mean(axis=1, skipna=False)
    m_ = m_[["a_code", "ym", name_]]
    PMX = m_ if PMX is None else PMX.merge(m_, on=["a_code", "ym"], how="outer")

D = (H[H.a_code.isin(TAM_CODES)].merge(PMX, on=["a_code", "ym"], how="left")
     .merge(MET[["ym", "temp", "rh", "precip"]], on="ym", how="inner")
     .merge(TAM[["a_code", "pop", "downwind", "road_grp"]], on="a_code"))
D = D[(D.ym >= START) & (D.ym <= END)]
if DROP_DECEMBER: D = D[~D.ym.str.endswith("-12")]
D = D.dropna(subset=list(LAGS)).sort_values(["a_code", "ym"]).reset_index(drop=True)
for g, cols in GROUPS.items(): D[g] = D[cols].sum(axis=1).round().astype(int)
D["tidx"] = D.a_code.map(TIDX); D["moy"] = pd.to_datetime(D.ym).dt.month
D["district"] = D.a_code.str[:4]
for v in ["temp", "rh", "precip"]: D["z_" + v] = (D[v] - D[v].mean()) / D[v].std()
print(f"{len(D):,} ตำบล-เดือน = {D.a_code.nunique()} ตำบล × {D.ym.nunique()} เดือน ({D.ym.min()} ถึง {D.ym.max()}) "
      f"| ปีงบ {sorted(D.fy.unique())} (เดือนต่อปีงบ: {D.groupby('fy').ym.nunique().to_dict()})")
print("จำนวนครั้งที่มารับบริการ:", {g: int(D[g].sum()) for g in GROUPS}, "| รวม", int(D[list(GROUPS)].sum().sum()))

# โครงข่าย Queen + เชื่อมเกาะ + BYM2 scaling
A = Queen.from_dataframe(TAM, use_index=False, silence_warnings=True).sparse.toarray().astype(float); A = ((A + A.T) > 0) * 1.0
cen = np.c_[TAM.to_crs(UTM).centroid.x, TAM.to_crs(UTM).centroid.y]
nc_, lab_ = connected_components(sparse.csr_matrix(A), directed=False)
while nc_ > 1:
    i_in, i_out = np.where(lab_ == 0)[0], np.where(lab_ != 0)[0]
    dd = np.sqrt(((cen[i_in, None] - cen[None, i_out]) ** 2).sum(-1)); a, b = np.unravel_index(dd.argmin(), dd.shape)
    A[i_in[a], i_out[b]] = A[i_out[b], i_in[a]] = 1; nc_, lab_ = connected_components(sparse.csr_matrix(A), directed=False)
Q = np.diag(A.sum(1)) - A
Qi = np.linalg.inv(Q + np.eye(N) * Q.diagonal().max() * np.sqrt(np.finfo(float).eps)); one = np.ones((1, N))
QC = Qi - Qi @ one.T @ np.linalg.inv(one @ Qi @ one.T) @ one @ Qi
SCALE = float(np.exp(np.mean(np.log(np.diag(QC))))); S_ICAR = QC / SCALE
print(f"BYM2: {N} ตำบล, เพื่อนบ้านเฉลี่ย {A.sum(1).mean():.2f}, connected components 1, scaling factor s = {SCALE:.3f}")

## 4b · Rate ratio ตามเขตสัมผัส (Sect. 2.7, 3.4)
Poisson regression ของจำนวนครั้งรายตำบล-เดือน, offset = ln ประชากร, **fiscal-year strata** (`C(fy)`), CI แบบ cluster-robust ตามตำบล
- PM2.5 Q4 vs Q1: ควอร์ไทล์ของค่าเฉลี่ย PM2.5 (lag 0) ของตำบลตลอดช่วงศึกษา (เฉพาะตำบล Q1 และ Q4)
- Downwind vs อื่น ๆ (ทุกตำบล) · Roadside high vs none (เฉพาะสองกลุ่มนี้)

In [ ]:
import statsmodels.api as sm, statsmodels.formula.api as smf
TAM["pm_mean"] = TAM.a_code.map(D.groupby("a_code")["Lag 0"].mean())
TAM["pm_q"] = pd.qcut(TAM.pm_mean, 4, labels=[1, 2, 3, 4]).astype(int)
D = D.merge(TAM[["a_code", "pm_q"]], on="a_code", how="left")

def rate_ratio(df, outcome, flag):
    d = df.assign(x=np.asarray(flag).astype(int), yy=df[outcome])
    r = smf.glm("yy ~ x + C(fy)", data=d, family=sm.families.Poisson(), offset=np.log(d["pop"])).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(d.a_code)[0]})
    b, se = r.params["x"], r.bse["x"]
    return dict(RR=np.exp(b), lo=np.exp(b - 1.96 * se), hi=np.exp(b + 1.96 * se), p=r.pvalues["x"],
                n_exposed=int(d.loc[d.x == 1, "a_code"].nunique()), n_reference=int(d.loc[d.x == 0, "a_code"].nunique()))

RR = pd.DataFrame()
if RUN_RR:
    rows = []
    for g in GROUPS:
        q = D[D.pm_q.isin([1, 4])]; rows.append(dict(outcome=g, contrast="PM2.5 Q4 vs Q1", **rate_ratio(q, g, q.pm_q == 4)))
        rows.append(dict(outcome=g, contrast="Downwind vs other", **rate_ratio(D, g, D.downwind == 1)))
        r_ = D[D.road_grp.isin(["none", "high"])]
        rows.append(dict(outcome=g, contrast="Roadside high vs none", **rate_ratio(r_, g, r_.road_grp == "high")))
    RR = pd.DataFrame(rows); RR.to_csv(f"{OUTDIR}/RR_exposure_zones.csv", index=False, encoding="utf-8-sig")
    display(RR.round(3))

## 5 · ฟังก์ชันแบบจำลอง
- `prep()` สร้างอาร์เรย์สำหรับแถวที่ใช้เรียน · ค่าเฉลี่ย PM2.5 ของตำบล ($\bar x_i$) คำนวณจากแถวเรียน (ใช้ในการทำนายช่วงสุดท้าย)
- `make_model()` — `fixed=None` สำหรับ MCMC · `fixed={sigma, rho, inv_alpha}` สำหรับ conditional posterior mode ในการทำนายนอกตัวอย่าง
- Effect modification: ความชัน PM2.5 ภายในตำบล = $\beta_W + \beta_D\,\text{downwind}_i + \beta_L\,\text{low}_i + \beta_H\,\text{high}_i$

In [ ]:
EYE = np.eye(N)
MOD_COLS = ["downwind", "road_low", "road_high"]
D["road_low"] = (D.road_grp == "low").astype(int); D["road_high"] = (D.road_grp == "high").astype(int)

def prep(df, outcome, lag, train_mask=None):
    """อาร์เรย์สำหรับแบบจำลอง; train_mask = แถวที่ใช้คำนวณค่าเฉลี่ยตำบลและระดับของผลเวลา"""
    tm = np.ones(len(df), bool) if train_mask is None else np.asarray(train_mask)
    rate = df[outcome].sum() / (df["pop"].sum())                       # อัตราต่อคน-เดือน ของทั้งจังหวัด
    mean_i = df[tm].groupby("tidx")[lag].mean().reindex(range(N))
    mean_i = mean_i.fillna(df.groupby("tidx")[lag].mean().reindex(range(N)))
    xbar = float(mean_i.mean())
    fy_levels = sorted(df.loc[tm, "fy"].unique())
    fy_idx = np.array([fy_levels.index(f) if f in fy_levels else len(fy_levels) - 1 for f in df.fy])   # ปีงบที่ไม่มีในชุดเรียน → ใช้ปีล่าสุดของชุดเรียน
    moy_levels = sorted(df.loc[tm, "moy"].unique())
    return dict(y=df[outcome].values, logE=np.log(df["pop"].values * rate), tidx=df.tidx.values,
                xw=((df[lag].values - mean_i.values[df.tidx.values]) / PM_UNIT), xb_i=(mean_i.values - xbar) / PM_UNIT,
                Z=df[["z_temp", "z_rh", "z_precip"]].values, moy=np.array([moy_levels.index(m) for m in df.moy]),
                fy=fy_idx, n_moy=len(moy_levels), n_fy=len(fy_levels), M=df[MOD_COLS].values.astype(float), rate=rate)

def make_model(a, rows, spatial=True, use_pm=True, mods=False, fixed=None):
    r = np.asarray(rows)
    with pm.Model() as m:
        b0 = pm.Normal("b0", 0, 2)
        gam = pm.Normal("gamma", 0, 1, shape=3)
        mu_i = b0
        if use_pm:
            b_w = pm.Normal("b_within", 0, 1); b_b = pm.Normal("b_between", 0, 1); mu_i = b0 + b_b * a["xb_i"]
        mu_i = mu_i + pt.zeros(N)
        if fixed is None:
            sigma = pm.HalfNormal("sigma", 1); inv_a = pm.HalfNormal("inv_alpha", 1)
            if spatial:
                rho = pm.Beta("rho", 0.5, 0.5)
                theta = pm.MvNormal("theta", mu=mu_i, cov=sigma ** 2 * ((1 - rho) * EYE + rho * S_ICAR) + 1e-6 * EYE)
            else:
                theta = pm.Normal("theta", mu_i, sigma, shape=N)
        else:
            inv_a = fixed["inv_alpha"]
            cov = fixed["sigma"] ** 2 * (((1 - fixed["rho"]) * EYE + fixed["rho"] * S_ICAR) if spatial else EYE) + 1e-6 * EYE
            theta = pm.MvNormal("theta", mu=mu_i, chol=np.linalg.cholesky(cov))
        pm.Deterministic("phi", theta - mu_i)
        d_m = pm.ZeroSumNormal("moy", sigma=1, shape=a["n_moy"])
        d_f = pm.ZeroSumNormal("fyear", sigma=1, shape=a["n_fy"]) if a["n_fy"] > 1 else pt.zeros(1)
        eta = a["logE"][r] + theta[a["tidx"][r]] + pt.dot(a["Z"][r], gam) + d_m[a["moy"][r]] + d_f[a["fy"][r]]
        if use_pm:
            slope = b_w
            if mods:
                b_mod = pm.Normal("b_mod", 0, 1, shape=len(MOD_COLS)); slope = b_w + pt.dot(a["M"][r], b_mod)
            eta = eta + slope * a["xw"][r]
        pm.NegativeBinomial("y", mu=pt.exp(eta), alpha=1 / inv_a, observed=a["y"][r])
    return m

HAS_NUMPYRO = importlib.util.find_spec("numpyro") is not None and importlib.util.find_spec("jax") is not None
CONV = {}
def fit(name, a, **kw):
    """MCMC พร้อม checkpoint; ถ้ามี divergence รันใหม่ด้วย target_accept 0.97"""
    f = os.path.join(FITDIR, f"{name}.nc")
    if os.path.exists(f):
        idata = az.from_netcdf(f); print(f"↺ {name}: โหลดจาก checkpoint")
    else:
        for ta in (TARGET_ACCEPT, TARGET_ACCEPT_RETRY):
            t0 = time.time()
            with make_model(a, np.arange(len(a["y"])), **kw):
                skw = dict(nuts_sampler="numpyro", nuts_sampler_kwargs={"nuts_kwargs": {"dense_mass": True}}) if HAS_NUMPYRO \
                      else dict(init="jitter+adapt_full")
                idata = pm.sample(DRAWS, tune=TUNE, chains=CHAINS, target_accept=ta, random_seed=SEED, progressbar=False,
                                  idata_kwargs={"log_likelihood": True}, **skw)
            div = int(idata.sample_stats["diverging"].sum())
            print(f"✓ {name}: {time.time() - t0:.0f} s, target_accept {ta}, divergences {div}", flush=True)
            if div == 0: break
        idata.posterior.attrs["target_accept"] = ta
        idata.to_netcdf(f)
    vars_ = [v for v in ["b0", "b_within", "b_between", "b_mod", "gamma", "sigma", "inv_alpha", "rho"] if v in idata.posterior]
    s = az.summary(idata, var_names=vars_)
    div = int(idata.sample_stats["diverging"].sum())
    CONV[name] = dict(max_rhat=s.r_hat.max(), min_ess_bulk=s.ess_bulk.min(), min_ess_tail=s.ess_tail.min(), divergences=div,
                      target_accept=float(idata.posterior.attrs.get("target_accept", np.nan)),
                      passed=bool(s.r_hat.max() <= 1.01 and s.ess_bulk.min() > 400 and s.ess_tail.min() > 400 and div == 0))
    return idata

def irr(idata, expr="b_within"):
    e = np.exp(idata.posterior[expr].values.ravel())
    return dict(IRR=np.median(e), lo=np.quantile(e, .025), hi=np.quantile(e, .975), P_gt1=(e > 1).mean())
fmt = lambda r: f"{r['IRR']:.2f} ({r['lo']:.2f}–{r['hi']:.2f})"

## 6 · Table 5 — IRR ต่อ PM2.5 10 µg/m³ จาก NB-BYM2 (4 กลุ่มโรค × 3 lag)
คอลัมน์ lag = IRR ภายในตำบล · คอลัมน์ between = IRR ระหว่างตำบลจากแบบจำลอง lag 0

In [ ]:
FITS, T5 = {}, []
if RUN_TABLE5:
    for g in GROUPS:
        for lag in LAGS:
            key = f"bym2_pm_{g}_{lag}".replace(" ", "").replace("–", "-")
            FITS[(g, lag)] = fit(key, prep(D, g, lag), spatial=True, use_pm=True)
            r = irr(FITS[(g, lag)]); T5.append(dict(outcome=g, term=lag, **r))
        T5.append(dict(outcome=g, term="Between-tambon", **irr(FITS[(g, "Lag 0")], "b_between")))
    T5 = pd.DataFrame(T5); T5.to_csv(f"{OUTDIR}/Table5_IRR_long.csv", index=False, encoding="utf-8-sig")
    TAB5 = T5.assign(v=T5.apply(fmt, axis=1)).pivot(index="outcome", columns="term", values="v")
    TAB5 = TAB5.reindex(index=list(GROUPS), columns=list(LAGS) + ["Between-tambon"])
    TAB5.to_csv(f"{OUTDIR}/Table5_IRR.csv", encoding="utf-8-sig")
    display(TAB5); display(T5.round(3))
    # Fig. 6 — IRR ภายในตำบลตามกลุ่มโรคและ lag
    fig, ax = plt.subplots(figsize=(6.5, 4)); W_ = T5[T5.term != "Between-tambon"].reset_index(drop=True)
    cols_ = dict(zip(LAGS, ["#2c7bb6", "#d7191c", "#1a9641"]))
    for j, lag in enumerate(LAGS):
        s_ = W_[W_.term == lag]; yy = np.arange(len(GROUPS))[::-1] + (1 - j) * 0.22
        ax.errorbar(s_.IRR, yy, xerr=[s_.IRR - s_.lo, s_.hi - s_.IRR], fmt="o", capsize=3, color=cols_[lag], label=lag)
    ax.set_yticks(np.arange(len(GROUPS))[::-1], list(GROUPS)); ax.axvline(1, ls="--", c="k")
    ax.set_xlabel("Within-tambon IRR per 10 µg/m³ PM$_{2.5}$ (95% CrI)"); ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout(); plt.savefig(f"{OUTDIR}/Fig6_IRR.png", dpi=300); plt.show()

## 7 · Table 4 — เปรียบเทียบแบบจำลอง (โรคระบบทางเดินหายใจ, lag 0)
| แบบจำลอง | โครงสร้างพื้นที่ | PM2.5 |
|---|---|---|
| NB, no PM2.5 | ผลสุ่มรายตำบลแบบอิสระ | – |
| NB-BYM2, no PM2.5 | BYM2 | – |
| NB-BYM2 + PM2.5 | BYM2 | within + between |
| NB-BYM2 + PM2.5 × modifiers | BYM2 | within × (downwind, roadside low, roadside high) + between |

ELPD จาก PSIS-LOO (Vehtari et al. 2017) · ผลต่างระหว่างแบบจำลองคำนวณแบบจับคู่รายจุด (Δelpd ± SE)

In [ ]:
T4_OUT, T4_LAG = "Respiratory", "Lag 0"
SPECS = {"NB, no PM2.5":                 dict(spatial=False, use_pm=False),
         "NB-BYM2, no PM2.5":            dict(spatial=True,  use_pm=False),
         "NB-BYM2 + PM2.5":              dict(spatial=True,  use_pm=True),
         "NB-BYM2 + PM2.5 × modifiers":  dict(spatial=True,  use_pm=True, mods=True)}
A4 = prep(D, T4_OUT, T4_LAG); F4, LOO = {}, {}
if RUN_TABLE4:
    for lab, kw in SPECS.items():
        if lab == "NB-BYM2 + PM2.5" and (T4_OUT, T4_LAG) in FITS: F4[lab] = FITS[(T4_OUT, T4_LAG)]; continue
        key = {"NB, no PM2.5": "nb_iid_nopm", "NB-BYM2, no PM2.5": "bym2_nopm", "NB-BYM2 + PM2.5": f"bym2_pm_{T4_OUT}_Lag0",
               "NB-BYM2 + PM2.5 × modifiers": "bym2_pm_mods"}[lab] + f"_{T4_OUT}"
        F4[lab] = fit(key, A4, **kw)
    for lab, idt in F4.items(): LOO[lab] = az.loo(idt, pointwise=True)
    def d_elpd(a, b):
        d = LOO[a].loo_i.values - LOO[b].loo_i.values; return d.sum(), np.sqrt(len(d) * d.var())
    T4 = pd.DataFrame({lab: {"elpd_LOO": l.elpd_loo, "SE": l.se, "p_LOO": l.p_loo,
                             "Pareto k > 0.7 (n)": int((l.pareto_k.values > 0.7).sum())} for lab, l in LOO.items()}).T
    DELTA = pd.DataFrame([dict(comparison=f"{a} vs {b}", delta_elpd=d_elpd(a, b)[0], SE=d_elpd(a, b)[1]) for a, b in [
        ("NB-BYM2, no PM2.5", "NB, no PM2.5"), ("NB-BYM2 + PM2.5", "NB-BYM2, no PM2.5"),
        ("NB-BYM2 + PM2.5 × modifiers", "NB-BYM2 + PM2.5")]])
    display(T4.round(1)); display(DELTA.round(2))

## 8 · Table 4 (ต่อ) — การทำนายนอกตัวอย่าง (Sect. 2.9)
- **LODO:** ตัดตำบลทั้งอำเภอทีละอำเภอ (13 folds) · **ช่วงสุดท้าย:** เรียน ม.ค. 2565 – `TRAIN_END` → ทำนาย ม.ค.–พ.ย. 2567
  ค่าเฉลี่ย PM2.5 ของตำบลและผลของปีงบมาจากชุดเรียน (ปีงบที่ไม่มีในชุดเรียน → ใช้ปีงบล่าสุดของชุดเรียน)
- แต่ละ fold ใช้ **conditional posterior mode** ของผลคงที่และผลรายตำบล โดยตรึง σ, ρ, 1/α ที่มัธยฐาน posterior ของข้อมูลทั้งหมด
  (เป็น cross-validation แบบประมาณ เพราะพารามิเตอร์ความแปรปรวนใช้ข้อมูลทั้งหมด)
- ตำบลที่ถูกตัดออกจะได้ $\theta_i$ จากการแจกแจงแบบมีเงื่อนไขของ BYM2 ตามตำบลข้างเคียง
- เกณฑ์: RMSE และ Spearman ของอัตราต่อ 1,000 คนต่อปีรายตำบล (ค่าจริง vs ทำนาย)

In [ ]:
def post_median(idt, v, default):
    return float(idt.posterior[v].median()) if v in idt.posterior else default

def predict_fold(lab, kw, train, test, mean_from_train):
    idt = F4[lab]
    fixed = dict(sigma=post_median(idt, "sigma", 1.0), rho=post_median(idt, "rho", 0.0), inv_alpha=post_median(idt, "inv_alpha", 0.25))
    a = prep(D, T4_OUT, T4_LAG, train_mask=train if mean_from_train else None)   # LODO: ค่าเฉลี่ยตำบลจากแถวของตำบลนั้นเอง
    with make_model(a, np.where(train)[0], fixed=fixed, **kw) as m:
        mp = pm.find_MAP(progressbar=False, maxeval=20000)
    eta = a["logE"] + mp["theta"][a["tidx"]] + a["Z"] @ mp["gamma"] + mp["moy"][a["moy"]] + \
          (mp["fyear"][a["fy"]] if "fyear" in mp else 0)
    if kw.get("use_pm"):
        slope = mp["b_within"] + (a["M"] @ mp["b_mod"] if kw.get("mods") else 0)
        eta = eta + slope * a["xw"]
    return np.exp(eta)[test]

def rate_metrics(obs_rows, mu):
    t = obs_rows.assign(mu=mu).groupby("a_code").agg(y=(T4_OUT, "sum"), mu=("mu", "sum"), pop=("pop", "first"), n=("ym", "nunique"))
    f = 1000 * 12 / (t["pop"] * t.n); ro, rp = t.y * f, t.mu * f
    return float(np.sqrt(((ro - rp) ** 2).mean())), float(spearmanr(ro, rp).correlation)

CV = {}
if RUN_CV and F4:
    last_test = (D.ym > TRAIN_END) & (D.ym >= f"{int(TRAIN_END[:4]) + 1}-01")
    for lab, kw in SPECS.items():
        t0 = time.time(); mu_lodo = np.full(len(D), np.nan)
        for dist in sorted(D.district.unique()):
            test = (D.district == dist).values
            mu_lodo[test] = predict_fold(lab, kw, ~test, test, mean_from_train=False)
        lodo = rate_metrics(D, mu_lodo)
        train = (D.ym <= TRAIN_END).values; test = last_test.values
        mu_last = predict_fold(lab, kw, train, test, mean_from_train=True)
        last = rate_metrics(D[test], mu_last)
        CV[lab] = {"LODO RMSE": lodo[0], "LODO Spearman": lodo[1], "Last-period RMSE": last[0], "Last-period Spearman": last[1]}
        print(f"✓ {lab}: LODO RMSE {lodo[0]:.1f}, ρs {lodo[1]:.2f} | last-period RMSE {last[0]:.1f}, ρs {last[1]:.2f} [{time.time() - t0:.0f} s]", flush=True)
    CV = pd.DataFrame(CV).T

if len(LOO):
    TABLE4 = T4.join(CV) if len(CV) else T4
    out4 = TABLE4.copy()
    out4["elpdLOO (SE)"] = [f"{e:,.1f} ({s:.1f})" for e, s in zip(TABLE4.elpd_LOO, TABLE4.SE)]
    cols4 = ["elpdLOO (SE)"] + [c for c in ["LODO RMSE", "LODO Spearman", "Last-period RMSE", "Last-period Spearman"] if c in out4]
    out4 = out4[cols4].round(2); out4.to_csv(f"{OUTDIR}/Table4_model_comparison.csv", encoding="utf-8-sig")
    DELTA.to_csv(f"{OUTDIR}/Table4_delta_elpd.csv", index=False, encoding="utf-8-sig")
    display(out4)

## 9 · Effect modification (Sect. 3.6, Fig. 7)
IRR ของแต่ละกลุ่ม = exp(ค่าเฉลี่ยของความชันรายแถวในกลุ่มนั้น) ต่อ posterior draw — กลุ่ม downwind จึงเฉลี่ยตามสัดส่วนกลุ่มริมถนนที่มีจริงในกลุ่มนั้น (และกลับกัน)

In [ ]:
EM = pd.DataFrame()
lab_m = "NB-BYM2 + PM2.5 × modifiers"
if lab_m in F4:
    po = F4[lab_m].posterior
    bw = po["b_within"].values.reshape(-1); bm = po["b_mod"].values.reshape(-1, len(MOD_COLS))
    S_ = bw[:, None] + bm @ A4["M"].T                                  # draws × แถว
    grp = {"Downwind": D.downwind.values == 1, "Not downwind": D.downwind.values == 0,
           "Roadside none": (D.road_grp == "none").values, "Roadside low": (D.road_grp == "low").values,
           "Roadside high": (D.road_grp == "high").values}
    gs = {k: S_[:, m].mean(1) for k, m in grp.items()}
    rows = []
    for k, s in gs.items():
        e = np.exp(s); rows.append(dict(group=k, n_tambons=int(D.loc[grp[k], "a_code"].nunique()), IRR=np.median(e),
                                        lo=np.quantile(e, .025), hi=np.quantile(e, .975), P_gt1=(e > 1).mean()))
    EM = pd.DataFrame(rows)
    PCOMP = {"P(downwind > not downwind)": (gs["Downwind"] > gs["Not downwind"]).mean(),
             "P(roadside low > none)": (gs["Roadside low"] > gs["Roadside none"]).mean(),
             "P(roadside high > none)": (gs["Roadside high"] > gs["Roadside none"]).mean()}
    EM.to_csv(f"{OUTDIR}/EM_effect_modification.csv", index=False, encoding="utf-8-sig")
    display(EM.round(3)); print({k: round(v, 2) for k, v in PCOMP.items()})
    fig, ax = plt.subplots(figsize=(6, 3)); yy = np.arange(len(EM))[::-1]
    ax.errorbar(EM.IRR, yy, xerr=[EM.IRR - EM.lo, EM.hi - EM.IRR], fmt="o", capsize=3, color="#2c7bb6")
    ax.set_yticks(yy, EM.group); ax.axvline(1, ls="--", c="k"); ax.set_xlabel("Within-tambon respiratory IRR per 10 µg/m³ (95% CrI)")
    plt.tight_layout(); plt.savefig(f"{OUTDIR}/Fig7_effect_modification.png", dpi=300); plt.show()

## 9b · Predicted risk, ตำบลเป้าหมาย (Table 6, Fig. 8) และ attributable visits (Sect. 2.10, 3.8)
ใช้ posterior ของ NB-BYM2 + PM2.5 (โรคระบบทางเดินหายใจ, lag 0) — ไม่ต้องรัน MCMC เพิ่ม
- ต่อ posterior draw: $\hat\mu_{it}$ = ค่าคาดหมายจากแบบจำลอง · $SIR_i=\sum_t\hat\mu_{it}/\sum_t E_{it}$ · $P(RR_i>1)$ = สัดส่วน draw ที่ $SIR_i>1$
- elevated: $P>0.8$ · low: $P<0.2$ · uncertain: ที่เหลือ (Richardson et al. 2004)
- ตำบลเป้าหมาย = elevated **และ** (PM2.5 Q4 หรือ downwind หรือ roadside high)
- $AV_{c}=\sum_{i,t}\hat\mu_{it}\big(1-IRR^{-\max(0,x_{it}-c)/10}\big)$, $c$ = 15 (มาตรฐานไทย) และ 5 µg/m³ (WHO)

In [ ]:
RISK, T6, AV = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
_main = FITS.get(("Respiratory", "Lag 0"))
if _main is None: _main = F4.get("NB-BYM2 + PM2.5")
if RUN_RISK and _main is not None:
    a_ = prep(D, "Respiratory", "Lag 0"); po = _main.posterior
    st = lambda v: po[v].values.reshape(-1, *po[v].shape[2:])
    th, gm, mo, bw = st("theta"), st("gamma"), st("moy"), st("b_within")
    eta = a_["logE"][None, :] + th[:, a_["tidx"]] + gm @ a_["Z"].T + mo[:, a_["moy"]] + bw[:, None] * a_["xw"][None, :]
    if "fyear" in po: eta = eta + st("fyear")[:, a_["fy"]]
    MU = np.exp(eta)                                                    # draws × แถว
    E_ = np.exp(a_["logE"])
    tid = a_["tidx"]; onehot = sparse.csr_matrix((np.ones(len(tid)), (np.arange(len(tid)), tid)), shape=(len(tid), N))
    SIRd = (onehot.T @ MU.T).T / (onehot.T @ E_)[None, :]              # draws × ตำบล
    obs = D.groupby("tidx").agg(y=("Respiratory", "sum"), n=("ym", "nunique")).reindex(range(N))
    RISK = TAM[["a_code", "name", "district_name", "pop", "pm_mean", "pm_q", "downwind", "road_grp"]].copy()
    RISK["SIR"] = np.median(SIRd, 0); RISK["SIR_lo"] = np.quantile(SIRd, .025, 0); RISK["SIR_hi"] = np.quantile(SIRd, .975, 0)
    RISK["P_RR_gt1"] = (SIRd > 1).mean(0)
    RISK["risk_class"] = np.where(RISK.P_RR_gt1 > EXCEED_HI, "elevated", np.where(RISK.P_RR_gt1 < EXCEED_LO, "low", "uncertain"))
    RISK["crude_rate"] = obs.y.values / (RISK["pop"].values * obs.n.values) * 12000
    RISK["district_seat"] = RISK.a_code.str[4:6] == "01"
    if HOSPITAL_CODES: RISK["hospital"] = RISK.a_code.isin([str(c).zfill(6) for c in HOSPITAL_CODES])
    RISK["priority"] = (RISK.risk_class == "elevated") & ((RISK.pm_q == 4) | (RISK.downwind == 1) | (RISK.road_grp == "high"))
    RISK = RISK.sort_values(["P_RR_gt1", "SIR"], ascending=False).reset_index(drop=True)
    RISK.to_csv(f"{OUTDIR}/Risk_all_tambons.csv", index=False, encoding="utf-8-sig")

    T6 = RISK[RISK.priority].copy()
    T6 = pd.DataFrame({"Tambon": T6.name, "District": T6.district_name,
                       "Predicted SIR (95% CrI)": [f"{a:.2f} ({b:.2f}–{c:.2f})" for a, b, c in zip(T6.SIR, T6.SIR_lo, T6.SIR_hi)],
                       "P(RR > 1)": T6.P_RR_gt1.round(2), "PM2.5 quartile": T6.pm_q, "Downwind": np.where(T6.downwind == 1, "Yes", "No"),
                       "Roadside group": T6.road_grp.str.capitalize(), "Crude respiratory rate per 1,000/yr": T6.crude_rate.round(0).astype(int)}
                      | ({"Hospital tambon": np.where(T6.hospital, "Yes", "No")} if HOSPITAL_CODES else {}))
    T6.to_csv(f"{OUTDIR}/Table6_priority_tambons.csv", index=False, encoding="utf-8-sig"); display(T6)

    # attributable visits (สมการ 3)
    x_ = D["Lag 0"].values; IRRd = np.exp(bw)
    rows = []
    for lab, c in AV_THRESHOLDS.items():
        k = np.maximum(0, x_ - c) / PM_UNIT
        av = (MU * (1 - IRRd[:, None] ** (-k[None, :]))).sum(1); pct = 100 * av / MU.sum(1)
        rows.append(dict(threshold=lab, cutoff=c, AV=np.median(av), lo=np.quantile(av, .025), hi=np.quantile(av, .975),
                         pct=np.median(pct), pct_lo=np.quantile(pct, .025), pct_hi=np.quantile(pct, .975)))
    AV = pd.DataFrame(rows); AV.to_csv(f"{OUTDIR}/Attributable_visits.csv", index=False, encoding="utf-8-sig"); display(AV.round(1))

    # Fig. 8 — (a) SIR, (b) exceedance probability, (c) ตำบลเป้าหมาย
    G = TAM[["a_code", "geometry"]].merge(RISK, on="a_code"); OTHER = BND[~BND.a_code.isin(TAM.a_code)]
    G["cls"] = np.where(G.priority, "Priority", np.where(G.risk_class == "elevated", "Elevated (other)",
                        np.where(G.risk_class == "low", "Low", "Uncertain")))
    fig, axs = plt.subplots(1, 3, figsize=(16, 6))
    for ax in axs: OTHER.plot(ax=ax, color="#d9d9d9", edgecolor="white", lw=0.3); ax.set_axis_off()
    m_ = float(np.abs(np.log2(G.SIR)).max())
    G.assign(l2=np.log2(G.SIR)).plot(column="l2", ax=axs[0], cmap="RdBu_r", vmin=-m_, vmax=m_, edgecolor="white", lw=0.3,
                                     legend=True, legend_kwds={"label": "log$_2$ predicted SIR", "shrink": 0.6})
    G.plot(column="P_RR_gt1", ax=axs[1], cmap="viridis", vmin=0, vmax=1, edgecolor="white", lw=0.3,
           legend=True, legend_kwds={"label": "P(RR > 1 | data)", "shrink": 0.6})
    pal = {"Priority": "#b2182b", "Elevated (other)": "#f4a582", "Uncertain": "#f7f7f7", "Low": "#92c5de"}
    for k, c in pal.items():
        s_ = G[G.cls == k]
        if len(s_): s_.plot(ax=axs[2], color=c, edgecolor="#636363", lw=0.3)
    from matplotlib.patches import Patch
    axs[2].legend(handles=[Patch(fc=c, ec="#636363", label=k) for k, c in pal.items()], loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8, frameon=False)
    for ax, t in zip(axs, ["(a) Predicted SIR", "(b) Exceedance probability", "(c) Priority tambons"]): ax.set_title(t, loc="left")
    plt.tight_layout(); plt.savefig(f"{OUTDIR}/Fig8_predicted_risk.png", dpi=300); plt.show()
    del MU, eta

## 10 · การลู่เข้าและตัวเลขสำหรับ manuscript

In [ ]:
CONVDF = pd.DataFrame(CONV).T; CONVDF.to_csv(f"{OUTDIR}/convergence.csv", encoding="utf-8-sig"); display(CONVDF.round(3))
if not CONVDF.passed.all(): print("⚠ บางแบบจำลองไม่ผ่านเกณฑ์ (R-hat ≤ 1.01, ESS > 400, ไม่มี divergence) — เพิ่ม DRAWS/TUNE แล้วลบไฟล์ .nc ของแบบจำลองนั้นก่อนรันใหม่")

L = []
L.append(f"Data: {D.a_code.nunique()} tambons, {D.ym.nunique()} months ({D.ym.min()} to {D.ym.max()}, December excluded), "
         f"{len(D):,} tambon-months, {int(D[list(GROUPS)].sum().sum()):,} visits "
         + ", ".join(f"{g} {int(D[g].sum()):,}" for g in GROUPS) + ".")
L.append(f"Fiscal years: {D.groupby('fy').ym.nunique().to_dict()} months per fiscal year (BE).")
L.append(f"Downwind tambons: {int(TAM.downwind.sum())} of {N}; roadside groups: {TAM.road_grp.value_counts().to_dict()}.")
L.append(f"MCMC: PyMC {pm.__version__}, NumPyro NUTS (dense mass), {CHAINS} chains × ({TUNE} tuning + {DRAWS} draws), seed {SEED}.")
if len(CONVDF):
    L.append(f"Convergence: max R-hat = {CONVDF.max_rhat.max():.3f}; min bulk ESS = {CONVDF.min_ess_bulk.min():.0f}; "
             f"min tail ESS = {CONVDF.min_ess_tail.min():.0f}; divergent transitions = {int(CONVDF.divergences.sum())}; "
             f"target acceptance raised to 0.97 in {int((CONVDF.target_accept > 0.9).sum())} model(s).")
if len(F4):
    ro = F4["NB-BYM2 + PM2.5"].posterior["rho"].values.ravel()
    L.append(f"Spatial share rho = {np.median(ro):.2f} (95% CrI {np.quantile(ro, .025):.2f}–{np.quantile(ro, .975):.2f}).")
    for _, r in DELTA.iterrows(): L.append(f"Delta elpd_LOO, {r.comparison}: {r.delta_elpd:.1f} (SE {r.SE:.1f}).")
if len(T5):
    for g in GROUPS:
        rr = T5[(T5.outcome == g)].set_index("term")
        L.append(f"{g}: lag 0 {fmt(rr.loc['Lag 0'])}, P(IRR>1) = {rr.loc['Lag 0', 'P_gt1']:.2f}; lag 1 {fmt(rr.loc['Lag 1'])}; "
                 f"lag 0–1 {fmt(rr.loc['Lag 0–1'])}; between {fmt(rr.loc['Between-tambon'])}.")
if len(EM):
    for _, r in EM.iterrows(): L.append(f"EM {r.group} (n = {r.n_tambons}): {fmt(r)}.")
    for k, v in PCOMP.items(): L.append(f"{k} = {v:.2f}.")
if len(RR):
    for _, r in RR.iterrows():
        L.append(f"Rate ratio {r.outcome}, {r.contrast}: {r.RR:.2f} (95% CI {r.lo:.2f}–{r.hi:.2f}), tambons {r.n_exposed} vs {r.n_reference}.")
if len(RISK):
    el = RISK[RISK.risk_class == "elevated"]
    L.append(f"Predicted respiratory SIR range {RISK.SIR.min():.3f}–{RISK.SIR.max():.1f}; elevated (P > {EXCEED_HI}) {len(el)}, "
             f"low (P < {EXCEED_LO}) {int((RISK.risk_class == 'low').sum())}, uncertain {int((RISK.risk_class == 'uncertain').sum())}.")
    L.append(f"Of the {len(el)} elevated: district-seat {int(el.district_seat.sum())}, downwind {int(el.downwind.sum())}, "
             f"PM2.5 Q4 {int((el.pm_q == 4).sum())}, roadside high {int((el.road_grp == 'high').sum())}"
             + (f", hospital tambons {int(el.hospital.sum())}" if HOSPITAL_CODES else "") + f"; priority tambons {int(RISK.priority.sum())}.")
if len(AV):
    for _, r in AV.iterrows():
        L.append(f"Attributable respiratory visits above {r.cutoff} µg/m³ ({r.threshold}): {r.AV:,.0f} (95% CrI {r.lo:,.0f} to {r.hi:,.0f}), "
                 f"{r.pct:.1f}% of predicted visits.")
TXT = "\n".join(L); print(TXT)
open(f"{OUTDIR}/manuscript_numbers.txt", "w", encoding="utf-8").write(TXT)
with pd.ExcelWriter(f"{OUTDIR}/Paper3_Table4_5.xlsx") as xw:
    if len(T5): TAB5.to_excel(xw, sheet_name="Table5"); T5.to_excel(xw, sheet_name="Table5_long", index=False)
    if len(LOO): out4.to_excel(xw, sheet_name="Table4"); DELTA.to_excel(xw, sheet_name="Delta_elpd", index=False)
    if len(EM): EM.to_excel(xw, sheet_name="Effect_modification", index=False)
    if len(RR): RR.to_excel(xw, sheet_name="RR_exposure_zones", index=False)
    if len(T6): T6.to_excel(xw, sheet_name="Table6", index=False); RISK.to_excel(xw, sheet_name="Risk_all_tambons", index=False)
    if len(AV): AV.to_excel(xw, sheet_name="Attributable_visits", index=False)
    CONVDF.to_excel(xw, sheet_name="Convergence")
print("\nบันทึกแล้วใน", OUTDIR)

## 11 · ข้อความ Methods ที่ต้องตรงกับโน้ตบุ๊กนี้
- **2.5:** "Month of year captured seasonality, and **fiscal-year (October–September)** effects captured changes in annual totals, including the COVID-19 disruption to service use in fiscal year 2565 (2022)."
- **2.7:** rate ratios ใช้ **fiscal-year strata** (ส่วน 4b ของโน้ตบุ๊กนี้ — นิยาม downwind เดียวกับ 2.4)
- **2.8:** "τ_fy(t) is the effect of the **fiscal year** containing month t" · priors: N(0, 2) intercept, N(0, 1) coefficients/interactions, half-normal(1) σ และ 1/α, Beta(0.5, 0.5) ρ, zero-sum normal(1) เดือนและปีงบ
- **2.9:** "…predicted January–November 2024, with tambon mean PM2.5 taken from the training period; the fiscal-year 2024 effect, estimated from October–November 2023, was also applied to October–November 2024 (fiscal year 2025)."
- **2.10:** ตรงกับส่วน 9b (SIR, exceedance 0.8/0.2, priority = elevated และ Q4/downwind/roadside high, AV ที่ 15 และ 5 µg/m³)